# Von Mises Plasticity — Interactive Notebook

## 1. Brief background

Most **ductile metals** yield according to the **von Mises criterion** (also called the
$J_2$ or *distortion-energy* criterion). The idea: a material starts to yield when the
**distortional** (shape-changing) strain energy reaches a critical value. Hydrostatic
pressure (equal squeezing in all directions) only changes volume, not shape, so it does
**not** cause yielding.

**Von Mises equivalent stress** (a single scalar that summarizes a full stress state):

$$\sigma_\text{vm}=\sqrt{\tfrac{1}{2}\left[(\sigma_1-\sigma_2)^2+(\sigma_2-\sigma_3)^2+(\sigma_3-\sigma_1)^2\right]}$$

or, in terms of the stress components,

$$\sigma_\text{vm}=\sqrt{\tfrac{1}{2}\left[(\sigma_{xx}-\sigma_{yy})^2+(\sigma_{yy}-\sigma_{zz})^2+(\sigma_{zz}-\sigma_{xx})^2\right]+3\left(\tau_{xy}^2+\tau_{yz}^2+\tau_{zx}^2\right)}$$

**Yield criterion:**

$$f=\sigma_\text{vm}-\sigma_Y \;\;\begin{cases}<0 & \text{elastic (inside the surface)}\\[2pt] =0 & \text{yielding (on the surface)}\\[2pt] >0 & \text{not admissible (outside)}\end{cases}$$

where $\sigma_Y$ is the current yield stress of the material.

### Plane-stress cross-section (the 2D graph)

In principal-stress space the von Mises surface is a **cylinder** whose axis is the
hydrostatic line $\sigma_1=\sigma_2=\sigma_3$. Taking a **plane-stress** slice
($\sigma_3=0$) gives an **ellipse**:

$$\sigma_1^2-\sigma_1\sigma_2+\sigma_2^2=\sigma_Y^2$$

Its major axis lies along $\sigma_1=\sigma_2$ (semi-axis $\sqrt{2}\,\sigma_Y$) and its
minor axis along $\sigma_1=-\sigma_2$ (semi-axis $\sqrt{2/3}\,\sigma_Y$). That ellipse is
what we plot and explore below.

### Hardening (why the sliders change the surface)

For a perfectly-plastic material the surface is fixed. With **isotropic hardening** the
surface *expands* as the material accumulates plastic strain $\varepsilon_p$:

$$\sigma_Y=\sigma_{y0}+H\,\varepsilon_p$$

where $\sigma_{y0}$ is the initial yield stress and $H$ is the (linear) hardening
modulus. The sliders below let you vary $\sigma_{y0}$, $H$ and $\varepsilon_p$ to see the
ellipse grow.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider
import plotly.graph_objects as go

# --- LaTeX-style typography (Computer Modern) for every matplotlib plot ---
# Computer Modern is LaTeX's default font, so plots match a LaTeX document
# without needing a TeX installation (no text.usetex, works on Binder as-is).
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['cmr10', 'CMU Serif', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',            # Computer Modern for $...$ math
    'mathtext.rm': 'serif',
    'axes.formatter.use_mathtext': True,
    'axes.unicode_minus': False,         # cmr10 has no unicode-minus glyph
    'figure.dpi': 110,
    'font.size': 12,
})

# Serif font stack for the Plotly 3D view (browser fonts, LaTeX-like).
PLOTLY_FONT = dict(family='CMU Serif, Latin Modern Roman, Georgia, Times New Roman, serif',
                   size=13)

## 2. Helper functions

* `von_mises_plane_stress` — equivalent stress for a 2D (plane-stress) state
  $(\sigma_{xx},\sigma_{yy},\tau_{xy})$:
  $\;\sigma_\text{vm}=\sqrt{\sigma_{xx}^2-\sigma_{xx}\sigma_{yy}+\sigma_{yy}^2+3\tau_{xy}^2}$
* `von_mises_3d` — equivalent stress from the three principal stresses.
* `principal_stresses` — Mohr's-circle principal values so we can plot the point in the
  $(\sigma_1,\sigma_2)$ plane.
* `yield_ellipse` — points on the von Mises ellipse for a given yield stress.

In [ ]:
def von_mises_plane_stress(sxx, syy, txy):
    """Von Mises equivalent stress for a plane-stress state (MPa)."""
    return np.sqrt(sxx**2 - sxx*syy + syy**2 + 3.0*txy**2)


def von_mises_3d(s1, s2, s3):
    """Von Mises equivalent stress from the three principal stresses (MPa)."""
    return np.sqrt(0.5*((s1 - s2)**2 + (s2 - s3)**2 + (s3 - s1)**2))


def principal_stresses(sxx, syy, txy):
    """In-plane principal stresses from Mohr's circle."""
    avg = 0.5*(sxx + syy)
    R = np.sqrt((0.5*(sxx - syy))**2 + txy**2)
    return avg + R, avg - R


def yield_ellipse(sigma_Y, n=400):
    """Points (s1, s2) on the von Mises plane-stress yield ellipse."""
    t = np.linspace(0.0, 2.0*np.pi, n)
    u = np.sqrt(2.0) * sigma_Y * np.cos(t)      # along the sigma1 = sigma2 axis
    v = np.sqrt(2.0/3.0) * sigma_Y * np.sin(t)  # along the sigma1 = -sigma2 axis
    s1 = (u + v)/np.sqrt(2.0)
    s2 = (u - v)/np.sqrt(2.0)
    return s1, s2

## 3. Interactive yield surface + stress-point checker (2D)

Use the sliders to:

* **Material properties** — change the initial yield stress $\sigma_{y0}$, the hardening
  modulus $H$, and the accumulated plastic strain $\varepsilon_p$. The current yield
  stress is $\sigma_Y=\sigma_{y0}+H\,\varepsilon_p$ and the ellipse resizes accordingly.
* **Stress point** — enter the plane-stress components $\sigma_{xx},\sigma_{yy},\tau_{xy}$.
  The notebook computes the principal stresses, plots the point at $(\sigma_1,\sigma_2)$,
  computes $\sigma_\text{vm}$, and tells you whether the state is **inside (elastic, green)**
  or **outside (yielding, red)** the surface — both numerically and graphically.

The title reports $\sigma_\text{vm}$, the status, the principal stresses, and the
**safety factor** $\sigma_Y/\sigma_\text{vm}$ (>1 means still elastic).

In [ ]:
def plot_von_mises(sigma_y0=250.0, H=0.0, eps_p=0.0,
                    sxx=180.0, syy=-90.0, txy=60.0):
    # current (hardened) yield stress
    sigma_Y = sigma_y0 + H*eps_p

    s1e, s2e = yield_ellipse(sigma_Y)
    s1, s2 = principal_stresses(sxx, syy, txy)
    svm = von_mises_plane_stress(sxx, syy, txy)
    inside = svm <= sigma_Y
    color = 'seagreen' if inside else 'crimson'

    fig, ax = plt.subplots(figsize=(7, 7))

    # yield surface
    ax.plot(s1e, s2e, color='steelblue', lw=2.2,
            label=f'Yield surface  $\\sigma_Y$={sigma_Y:.0f} MPa')
    ax.fill(s1e, s2e, color='steelblue', alpha=0.08)

    # reference axes / hydrostatic line
    lim = max(np.max(np.abs(np.concatenate([s1e, s2e])))*1.15,
              abs(s1)*1.2, abs(s2)*1.2, 1.0)
    ax.axhline(0, color='0.6', lw=0.8)
    ax.axvline(0, color='0.6', lw=0.8)
    ax.plot([-lim, lim], [-lim, lim], '--', color='0.7', lw=0.8,
            label='hydrostatic axis  $\\sigma_1=\\sigma_2$')

    # stress point
    ax.plot([0, s1], [0, s2], ls=':', color=color, lw=1.2)
    ax.plot(s1, s2, 'o', color=color, ms=12, zorder=5,
            markeredgecolor='k', markeredgewidth=0.6)
    ax.annotate(f'  ({s1:.0f}, {s2:.0f})', (s1, s2), color=color,
                fontsize=10, va='center')

    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_xlabel(r'$\sigma_1$  (MPa)')
    ax.set_ylabel(r'$\sigma_2$  (MPa)')
    ax.grid(alpha=0.3)
    ax.legend(loc='upper left', fontsize=9)

    status = 'INSIDE  (elastic)' if inside else 'OUTSIDE  (yielding)'
    sf = sigma_Y/svm if svm > 0 else np.inf
    ax.set_title(
        f'$\\sigma_{{vm}}$ = {svm:.1f} MPa   $\\rightarrow$   {status}\n'
        f'principal: $\\sigma_1$={s1:.1f}, $\\sigma_2$={s2:.1f} MPa'
        f'    |    safety factor = {sf:.2f}',
        color=color, fontsize=12)
    plt.show()


interact(
    plot_von_mises,
    sigma_y0=FloatSlider(value=250, min=50,  max=600, step=10,
                         description='σ_y0 (MPa)', style={'description_width':'initial'}),
    H=FloatSlider(value=0, min=0, max=5000, step=100,
                  description='Hardening H (MPa)', style={'description_width':'initial'}),
    eps_p=FloatSlider(value=0, min=0, max=0.10, step=0.005, readout_format='.3f',
                      description='plastic strain ε_p', style={'description_width':'initial'}),
    sxx=FloatSlider(value=180, min=-600, max=600, step=10,
                    description='σ_xx (MPa)', style={'description_width':'initial'}),
    syy=FloatSlider(value=-90, min=-600, max=600, step=10,
                    description='σ_yy (MPa)', style={'description_width':'initial'}),
    txy=FloatSlider(value=60, min=-400, max=400, step=10,
                    description='τ_xy (MPa)', style={'description_width':'initial'}),
);

## 4. Quick numeric check (no sliders)

If you just want to test a single stress state against a fixed yield stress, edit the
values below and run the cell.

In [ ]:
# --- edit these ---
sigma_Y = 250.0          # yield stress (MPa)
sxx, syy, txy = 200.0, -100.0, 80.0   # plane-stress components (MPa)
# ------------------

svm = von_mises_plane_stress(sxx, syy, txy)
s1, s2 = principal_stresses(sxx, syy, txy)

print(f'Principal stresses : sigma_1 = {s1:8.2f} MPa,  sigma_2 = {s2:8.2f} MPa')
print(f'Von Mises stress   : sigma_vm = {svm:7.2f} MPa')
print(f'Yield stress       : sigma_Y  = {sigma_Y:7.2f} MPa')
print(f'Safety factor      : {sigma_Y/svm:6.3f}')
print('Result             :',
      'INSIDE  -> elastic' if svm <= sigma_Y else 'OUTSIDE -> yielding')

## 5. Uniaxial tensile test $\leftrightarrow$ von Mises equivalent stress

Where do $\sigma_{y0}$ and $H$ *come from*? From a **uniaxial tensile test** — the single
experiment that calibrates the whole multiaxial model.

In a pure tension test only $\sigma_{xx}=\sigma$ is non-zero, so the von Mises stress
reduces to the axial stress itself:

$$\sigma_\text{vm}=\sqrt{\sigma^2}=\sigma .$$

This is the key link: **the axial stress in the tensile test *is* the von Mises
equivalent stress**, and the stress at which the test specimen yields *is* the radius
$\sigma_Y$ of the multiaxial von Mises surface.

We model the material as **bilinear (elastic – linear isotropic hardening)**:

$$\sigma=\begin{cases}E\,\varepsilon & \varepsilon\le\varepsilon_y=\sigma_{y0}/E \quad(\text{elastic})\\[4pt]
\dfrac{\sigma_{y0}+H\,\varepsilon}{1+H/E} & \varepsilon>\varepsilon_y \quad(\text{plastic})\end{cases}$$

The total strain splits as $\varepsilon=\varepsilon_e+\varepsilon_p=\sigma/E+\varepsilon_p$,
the plastic tangent modulus is $E_t=\dfrac{EH}{E+H}$, and the current stress equals the
current yield radius $\sigma_Y=\sigma_{y0}+H\,\varepsilon_p$.

**Left panel** — the stress–strain curve (drag the strain slider to "load" the specimen).
**Right panel** — the *same* number shown as the von Mises plane-stress surface: as you
pull past yield, the surface (initially dashed) **expands** to the current $\sigma_Y$, and
the loading point sits right on it at $(\sigma,0)$.

In [ ]:
def plot_stress_strain(E_GPa=200.0, sigma_y0=250.0, H=2000.0, eps_applied=0.020):
    E = E_GPa*1.0e3                     # Young's modulus in MPa
    eps_y = sigma_y0/E                  # yield strain
    E_t = E*H/(E + H) if (E + H) > 0 else 0.0   # plastic tangent modulus

    # bilinear stress-strain curve
    eps = np.linspace(0.0, max(eps_applied, 1.2*eps_y), 400)
    sig = np.where(E*eps <= sigma_y0,
                   E*eps,
                   (sigma_y0 + H*eps)/(1.0 + H/E))

    # state at the applied strain
    if E*eps_applied <= sigma_y0:
        sig_now = E*eps_applied
    else:
        sig_now = (sigma_y0 + H*eps_applied)/(1.0 + H/E)
    eps_p = max(eps_applied - sig_now/E, 0.0)   # plastic strain
    sigma_Y = sigma_y0 + H*eps_p                 # current yield radius (= sig_now if plastic)
    yielding = eps_applied > eps_y

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.6))

    # ---------------- LEFT: uniaxial stress-strain ----------------
    axL.plot(eps, sig, color='steelblue', lw=2.4)
    axL.plot(eps_y, sigma_y0, 'o', color='0.35', ms=7)
    axL.annotate(f'  yield  ($\\varepsilon_y$={eps_y:.4f},\n  $\\sigma_{{y0}}$={sigma_y0:.0f})',
                 (eps_y, sigma_y0), color='0.35', fontsize=9, va='top')

    pcol = 'crimson' if yielding else 'seagreen'
    axL.plot(eps_applied, sig_now, 'o', color=pcol, ms=12,
             markeredgecolor='k', markeredgewidth=0.6, zorder=5)
    axL.plot([eps_applied, eps_applied], [0, sig_now], ls=':', color=pcol, lw=1)
    axL.plot([0, eps_applied], [sig_now, sig_now], ls=':', color=pcol, lw=1)

    # illustrate elastic vs plastic strain split when yielding
    if yielding:
        axL.plot([sig_now/E, eps_applied], [sig_now, sig_now], color='darkorange',
                 lw=3, solid_capstyle='butt')
        axL.annotate(f'$\\varepsilon_p$={eps_p:.4f}',
                     (0.5*(sig_now/E + eps_applied), sig_now), color='darkorange',
                     fontsize=9, ha='center', va='bottom')

    axL.set_xlim(0, eps[-1])
    axL.set_ylim(0, max(sig.max(), sig_now)*1.15)
    axL.set_xlabel(r'strain  $\varepsilon$  (–)')
    axL.set_ylabel(r'stress  $\sigma = \sigma_\mathrm{vm}$  (MPa)')
    axL.grid(alpha=0.3)
    axL.set_title(f'Uniaxial tensile test\nE={E_GPa:.0f} GPa,  $E_t$={E_t:.0f} MPa,  '
                  f'$\\sigma$={sig_now:.0f} MPa', fontsize=11)

    # ---------------- RIGHT: matching von Mises surface ----------------
    s1i, s2i = yield_ellipse(sigma_y0)      # initial surface
    s1c, s2c = yield_ellipse(sigma_Y)       # current (expanded) surface
    axR.plot(s1i, s2i, '--', color='0.6', lw=1.6, label=f'initial $\\sigma_Y$={sigma_y0:.0f}')
    axR.plot(s1c, s2c, color='steelblue', lw=2.2, label=f'current $\\sigma_Y$={sigma_Y:.0f}')
    axR.fill(s1c, s2c, color='steelblue', alpha=0.07)

    lim = max(np.abs(s1c).max(), np.abs(s2c).max())*1.2
    axR.axhline(0, color='0.6', lw=0.8)
    axR.axvline(0, color='0.6', lw=0.8)
    axR.plot([-lim, lim], [-lim, lim], '--', color='0.8', lw=0.7)

    # uniaxial loading point sits at (sigma, 0) — on the current surface
    axR.plot([0, sig_now], [0, 0], ls=':', color=pcol, lw=1.2)
    axR.plot(sig_now, 0, 'o', color=pcol, ms=12, markeredgecolor='k',
             markeredgewidth=0.6, zorder=5)
    axR.annotate(f'  uniaxial\n  ({sig_now:.0f}, 0)', (sig_now, 0), color=pcol, fontsize=9)

    axR.set_xlim(-lim, lim)
    axR.set_ylim(-lim, lim)
    axR.set_aspect('equal')
    axR.set_xlabel(r'$\sigma_1$  (MPa)')
    axR.set_ylabel(r'$\sigma_2$  (MPa)')
    axR.grid(alpha=0.3)
    axR.legend(loc='upper left', fontsize=9)
    axR.set_title(r'Same state on the von Mises surface'
                  '\n' r'$\sigma_\mathrm{vm}=\sigma=\sigma_Y$ (on the surface)', fontsize=11)

    plt.tight_layout()
    plt.show()


interact(
    plot_stress_strain,
    E_GPa=FloatSlider(value=200, min=50, max=400, step=10,
                      description='E (GPa)', style={'description_width':'initial'}),
    sigma_y0=FloatSlider(value=250, min=50, max=600, step=10,
                         description='σ_y0 (MPa)', style={'description_width':'initial'}),
    H=FloatSlider(value=2000, min=0, max=10000, step=100,
                  description='Hardening H (MPa)', style={'description_width':'initial'}),
    eps_applied=FloatSlider(value=0.020, min=0.0005, max=0.05, step=0.0005,
                            readout_format='.4f', description='applied strain ε',
                            style={'description_width':'initial'}),
);

## 6. 3D von Mises yield surface (rotate with the mouse)

Finally, the full picture. In the principal-stress space $(\sigma_1,\sigma_2,\sigma_3)$
the von Mises surface is an **infinite circular cylinder** whose axis is the hydrostatic
line $\sigma_1=\sigma_2=\sigma_3$. Its radius is

$$\rho=\sqrt{\tfrac{2}{3}}\,\sigma_Y .$$

Everything **inside** the cylinder is elastic; the surface itself is where yielding
begins. The plane-stress ellipse from Section 3 is simply the slice of this cylinder by
the plane $\sigma_3=0$.

* **Drag with the mouse to rotate**, scroll to zoom (Plotly 3D).
* The **sliders** change the material ($\sigma_{y0},H,\varepsilon_p\Rightarrow\sigma_Y$)
  so the cylinder grows/shrinks, and let you place a **stress point**
  $(\sigma_1,\sigma_2,\sigma_3)$ that turns **green inside** / **red outside** the surface.
* Rotating the view does **not** reset when you move a slider (the figure updates in place).

In [ ]:
# orthonormal frame: n = hydrostatic axis, e1/e2 span the deviatoric plane
_n  = np.array([1.0, 1.0, 1.0])/np.sqrt(3.0)
_e1 = np.array([1.0, -1.0, 0.0])/np.sqrt(2.0)
_e2 = np.array([1.0, 1.0, -2.0])/np.sqrt(6.0)


def vm_cylinder(sigma_Y, half_len, n_theta=64):
    """Mesh (X,Y,Z) of the von Mises cylinder of radius sqrt(2/3)*sigma_Y."""
    rho = np.sqrt(2.0/3.0)*sigma_Y
    th = np.linspace(0.0, 2.0*np.pi, n_theta)
    h = np.array([-half_len, half_len])
    TH, HH = np.meshgrid(th, h)
    X = HH*_n[0] + rho*(np.cos(TH)*_e1[0] + np.sin(TH)*_e2[0])
    Y = HH*_n[1] + rho*(np.cos(TH)*_e1[1] + np.sin(TH)*_e2[1])
    Z = HH*_n[2] + rho*(np.cos(TH)*_e1[2] + np.sin(TH)*_e2[2])
    return X, Y, Z


def _axis_line(vec, L):
    return dict(x=[-L*vec[0], L*vec[0]], y=[-L*vec[1], L*vec[1]],
               z=[-L*vec[2], L*vec[2]])


# ---- sliders ----
w = dict(style={'description_width': 'initial'}, continuous_update=False)
sl_sy0  = FloatSlider(value=250, min=50, max=600, step=10, description='σ_y0 (MPa)', **w)
sl_H    = FloatSlider(value=0,   min=0, max=5000, step=100, description='Hardening H (MPa)', **w)
sl_epsp = FloatSlider(value=0,   min=0, max=0.10, step=0.005, readout_format='.3f',
                      description='plastic strain ε_p', **w)
sl_s1 = FloatSlider(value=200, min=-600, max=600, step=10, description='σ_1 (MPa)', **w)
sl_s2 = FloatSlider(value=-50, min=-600, max=600, step=10, description='σ_2 (MPa)', **w)
sl_s3 = FloatSlider(value=100, min=-600, max=600, step=10, description='σ_3 (MPa)', **w)

# ---- initial figure ----
_sigma_Y0 = sl_sy0.value + sl_H.value*sl_epsp.value
_L0 = 2.6*max(_sigma_Y0, 1.0)
X0, Y0, Z0 = vm_cylinder(_sigma_Y0, _L0)

fig3d = go.FigureWidget(layout=go.Layout(
    width=760, height=620, margin=dict(l=0, r=0, t=40, b=0),
    font=PLOTLY_FONT,
    scene=dict(xaxis_title='σ1 (MPa)', yaxis_title='σ2 (MPa)', zaxis_title='σ3 (MPa)',
               aspectmode='cube',
               camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))),
    title='von Mises yield cylinder'))

# trace 0: cylinder surface
fig3d.add_surface(x=X0, y=Y0, z=Z0, opacity=0.35, showscale=False,
                  colorscale=[[0, 'steelblue'], [1, 'steelblue']],
                  name='yield surface', hoverinfo='skip')
# trace 1: hydrostatic axis
ax_h = _axis_line(_n, _L0)
fig3d.add_scatter3d(x=ax_h['x'], y=ax_h['y'], z=ax_h['z'], mode='lines',
                    line=dict(color='black', width=4, dash='dash'),
                    name='hydrostatic axis', hoverinfo='skip')
# traces 2,3,4: principal axes
for vec, nm, col in [(np.array([1,0,0.]), 'σ1', 'firebrick'),
                     (np.array([0,1,0.]), 'σ2', 'green'),
                     (np.array([0,0,1.]), 'σ3', 'royalblue')]:
    a = _axis_line(vec, _L0)
    fig3d.add_scatter3d(x=a['x'], y=a['y'], z=a['z'], mode='lines',
                        line=dict(color=col, width=2), name=nm, hoverinfo='skip')
# trace 5: stress point
fig3d.add_scatter3d(x=[sl_s1.value], y=[sl_s2.value], z=[sl_s3.value],
                    mode='markers', marker=dict(size=6, color='seagreen'),
                    name='stress point')


def _update3d(change=None):
    sigma_Y = sl_sy0.value + sl_H.value*sl_epsp.value
    L = 2.6*max(sigma_Y, abs(sl_s1.value), abs(sl_s2.value), abs(sl_s3.value), 1.0)
    X, Y, Z = vm_cylinder(sigma_Y, L)
    svm = von_mises_3d(sl_s1.value, sl_s2.value, sl_s3.value)
    inside = svm <= sigma_Y
    with fig3d.batch_update():
        fig3d.data[0].x, fig3d.data[0].y, fig3d.data[0].z = X, Y, Z
        ah = _axis_line(_n, L)
        fig3d.data[1].x, fig3d.data[1].y, fig3d.data[1].z = ah['x'], ah['y'], ah['z']
        for i, vec in zip((2, 3, 4), (np.array([1,0,0.]), np.array([0,1,0.]), np.array([0,0,1.]))):
            a = _axis_line(vec, L)
            fig3d.data[i].x, fig3d.data[i].y, fig3d.data[i].z = a['x'], a['y'], a['z']
        fig3d.data[5].x = [sl_s1.value]
        fig3d.data[5].y = [sl_s2.value]
        fig3d.data[5].z = [sl_s3.value]
        fig3d.data[5].marker.color = 'seagreen' if inside else 'crimson'
        status = 'INSIDE (elastic)' if inside else 'OUTSIDE (yielding)'
        fig3d.layout.title = (f'σ_Y={sigma_Y:.0f} MPa   |   σ_vm={svm:.0f} MPa   '
                              f'→   {status}')


for _s in (sl_sy0, sl_H, sl_epsp, sl_s1, sl_s2, sl_s3):
    _s.observe(_update3d, names='value')
_update3d()

_controls = widgets.VBox([
    widgets.HTML('<b>Material</b>'), sl_sy0, sl_H, sl_epsp,
    widgets.HTML('<b>Stress point</b>'), sl_s1, sl_s2, sl_s3])
display(widgets.HBox([_controls, fig3d]))